# មេរៀន 05 - Agentic RAG


## ការរៀបចំ

កំណត់ត្រានេះបង្ហាញលំនាំ Agentic RAG (Retrieval-Augmented Generation) ដោយប្រើ Microsoft Agent Framework។

**អ្វីដែលត្រូវមាន:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ចំណុចបញ្ចប់សេវា Azure AI Search របស់អ្នក
- `AZURE_SEARCH_API_KEY` — កូដ API សម្រាប់ Azure AI Search របស់អ្នក
- ការដាក់ចេញ Azure OpenAI ដែលបានកំណត់តាមអថេរបរិស្ថាន
- Azure CLI ត្រូវបានផ្ទៀងផ្ទាត់ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG ជាអ្វី?

RAG តាមប្រពៃណីអនុវត្តតាមសង្វាក់កំណត់មួយ៖ ទាញយកឯកសារ រួចបង្កើតចម្លើយ។ **Agentic RAG** ទៅឆ្ងាយជាងនេះដោយផ្ដល់ឲ្យភ្នាក់ងារមានអំណាចសម្រេចចិត្ត **ពេលណា** និង **របៀបណា** ក្នុងការទាញយកព័ត៌មាន។

ជាមួយ Agentic RAG ភ្នាក់ងារអាចៈ
- **សម្រេចចិត្ត** ថាតើត្រូវការទាញយកព័ត៌មានមុនពេលឆ្លើយសំណួរ
- **ជ្រើសរើស** ប្រភពទិន្នន័យ ឬឧបករណ៍ដែលត្រូវសួរព័ត៌មាន
- **វាយតម្លៃ** លទ្ធផលដែលបានទាញយក ហើយអនុវត្តការទាញយកបន្ត ប្រសិនបើការព្យាយាមដំបូងមិនគ្រប់គ្រាន់
- **ផ្សំ** ព័ត៌មានពីជំហានទាញយកជាច្រើនទៅជាចម្លើយដែលសមរម្យ

នេះធ្វើឲ្យភ្នាក់ងារមានភាពបត់បែន និងត្រឹមត្រូវច្រើនជាងប្រព័ន្ធ retrieve-then-generate ដែលថេរ។


## បង្កើតឧបករណ៍ស្វែងរក

ក្នុង Agentic RAG, ប្រភពទិន្នន័យខាងក្រៅត្រូវបានរុំជា **ឧបករណ៍** ដែលភ្នាក់ងារអាចហៅបានតាមតម្រូវការ។ នេះអនុញ្ញាតឲ្យភ្នាក់ងារ មើលការទាញយកជាសកម្មភាពមួយទៀតដែលវាអាចអនុវត្តបាន ជំនួសមិនត្រូវធ្វើវាជាជំហានចាំបាច់។

ខាងក្រោមយើងបានកំណត់មូលដ្ឋានចំណេះដឹងពីការធ្វើដំណើរ ហើយបង្ហាញវាជាឧបករណ៍ដែលភ្នាក់ងារអាចហៅដើម្បីស្វែងរកព័ត៌មានអំពីកន្លែងគោលដៅ។


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## ការបង្កើតភ្នាក់ងារ RAG

ឥឡូវនេះ យើងបង្កើតភ្នាក់ងារ ដែលត្រូវបានណែនាំឱ្យ **តែងតែទាញយកព័ត៌មានមុនពេលឆ្លើយ**។ ភ្នាក់ងារនេះប្រើឧបករណ៍ `search_travel_knowledge` ដើម្បីឲ្យចម្លើយរបស់វាផ្អែកលើមូលដ្ឋានចំណេះដឹង មិនមែនពឹងផ្អែកលើទិន្នន័យបណ្តុះបណ្តាលរបស់ខ្លួនឯងទេ។


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ការទាញយកជាច្រើនដង — លំនាំ Maker-Checker

អត្ថប្រយោជន៍សំខាន់មួយនៃ Agentic RAG គឺ **ការទាញយកជាច្រើនដង**។ ភ្នាក់ងារ​អាចធ្វើការស្វែងរកជាច្រើនជុំ ដើម្បីបញ្ជាក់ កែតម្រូវ ឬពង្រីកលើលទ្ធផលដំបូងរបស់វា — ស្រដៀងនឹងដំណើរការ "maker-checker":

1. **ជំហាន Maker**: ភ្នាក់ងារទាញយកព័ត៌មានដំបូង និងរៀបចំចម្លើយមួយ។
2. **ជំហាន Checker**: ភ្នាក់ងារធ្វើការទាញព័ត៌មានបន្ថែម ដើម្បីបញ្ជាក់ព័ត៌មានលម្អិត ឬបំពេញចន្លោះ។

ខាងក្រោម ភ្នាក់ងារត្រូវបានសួរសំណួរមួយដែលទាមទារ​ឲ្យប្រៀបធៀបគោលដៅច្រើន ហើយបណ្តាលឲ្យវាស្វែងរកជាច្រើនដង។


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## សេចក្ដីសង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀបបង្កើតប្រព័ន្ធ **Agentic RAG** ដោយប្រើ Microsoft Agent Framework:

- **Agentic RAG** អនុញ្ញាតឱ្យភ្នាក់ងារសម្រេចចិត្តដោយឯករាជ្យថាតើយើងគួរទាញព័ត៌មាននៅពេលណា បង្កើតឱ្យការទាញយកមានភាពឌីណាមិច មិនមែនថេរឡើយ។
- **Tools as data sources**: មូលដ្ឋានចំណេះដឹងខាងក្រៅ (ដូចជា Azure AI Search) ត្រូវបានបញ្ចុះជាឧបករណ៍ដែលភ្នាក់ងារអាចហៅប្រើ។
- **Iterative retrieval**: លំនាំ maker-checker ធ្វើឱ្យភ្នាក់ងារអាចអនុវត្តការទាញយកច្រើនជុំ — ស្វែងរក, ផ្ទៀងផ្ទាត់, និងកែលម្អ — មុននឹងផ្តល់ចម្លើយចុងក្រោយ។

នៅក្នុងបរិបទផលិតកម្ម អ្នកនឹងជំនួស `TRAVEL_KNOWLEDGE_BASE` ដែលនៅក្នុងអង្គចងចាំ ជាមួយ Azure AI Search index ពិតប្រាកដ ដើម្បីគ្រប់គ្រងការទាញយកឯកសារច្រើនទាក់ទងនឹងការធ្វើដំណើរយ៉ាងទូលំទូលាយ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator). ខណៈដែលយើងខិតខំសម្រាប់ភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសាមាតុភូមិគួរត្រូវបានគេចាត់ទុកថាជាប្រភពផ្លូវការដែលអាចទុកចិត្តបាន។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមផ្តល់អនុសាសន៍ឱ្យប្រើការបកប្រែដោយអ្នកបកប្រែជាមនុស្សដែលមានវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកប្រែខុសដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
